# Experiment 4. Effect of Time Horizon on UP Performance

In [ ]:
using CSV
using DataFrames
using LinearAlgebra
using Plots
using Revise
include("../src/src.jl")
df= CSV.read("../dataset/prices_6stocks_20years.csv", DataFrame)
stocks_grid= ["BA", "PG", "KO", "WMT"]
prices_grid= Matrix{Float64}(df[:,stocks_grid])


# We fix eta this time and change time horizon
eta_T= 0.1
n,m= size(prices_grid)
T_list=[250,500,1000,2000,3000,4000,5000]
T_list=[T for T in T_list if T + 1 <= n]
T_results = DataFrame(T = Int[],UP_Final_Wealth = Float64[],
BCRP_Final_Wealth = Float64[],Total_Log_Regret = Float64[],
Average_Log_Regret = Float64[],Runtime_Seconds = Float64[])


for T in T_list
    prices_T= prices_grid[1:(T+1), :]
    runtime= @elapsed begin
        # We only need the wealth sequence here, ignore other variables.
        up_wealth_T,_, _=uni_port_func(prices_T, eta_T)
        # Compute BCRP.
        _,bcrp_final,_=bcrp_func_md(prices_T, eta_T)
    end 
    # final wealth of UP
    up_final= up_wealth_T[end]
    total_regret= log(bcrp_final)-log(up_final) # Log regret
    push!(T_results,(T,up_final,bcrp_final,total_regret,total_regret/T,runtime))
end

CSV.write("time_horizon_results.csv", T_results)
println(T_results)

# Plot average log regret 
plot(T_results.T,T_results.Average_Log_Regret,marker= :circle,
xlabel = "Time T",ylabel = "Average log regret",
title = "Average Log Regret vs Time Horizon",legend = false)
savefig("average_log_regret_vs_T.png")
# total log regret plot.
plot(T_results.T, T_results.Total_Log_Regret,marker = :circle,
xlabel = "Time horizon T",ylabel="Total log regret",
title = "Total Log Regret vs Time Horizon",legend = false)
savefig("total_log_regret_vs_T_real.png")
# Runtime
plot(T_results.T,T_results.Runtime_Seconds,marker = :circle,
xlabel= "Time horizon T",ylabel = "Runtime seconds",
title= "Runtime vs Time Horizon",legend= false)
savefig("runtime_vs_T_real.png")



7×6 DataFrame
 Row │ T      UP_Final_Wealth  BCRP_Final_Wealth  Total_Log_Regret  Average_Log_Regret  Runtime_Seconds 
     │ Int64  Float64          Float64            Float64           Float64             Float64         
─────┼──────────────────────────────────────────────────────────────────────────────────────────────────
   1 │   250          1.13305            1.26122          0.107166         0.000428664        0.023668
   2 │   500          1.27222            1.52249          0.179587         0.000359174        0.0307806
   3 │  1000          1.12916            1.40024          0.215169         0.000215169        0.0848785
   4 │  2000          1.82927            2.05094          0.11438          5.719e-5           0.628258
   5 │  3000          2.53309            3.85264          0.419319         0.000139773        0.160503
   6 │  4000          3.33638            3.90849          0.158263         3.95656e-5         0.136122
   7 │  5000          4.53484            6.86657   

"runtime_vs_T_real.png"